# NoLA on EuroSAT RGB: from images to prompted CLIP
Use the **Robotics Research Lab** kernel. The default is a small walkthrough, not paper accuracy. Real images are drawn from the published CoOp split. Follow [`NoLA.py`](upstream/trainers/NoLA.py): `gen_emb()` → `forward_with_text_desc()` → `select_top_k()` → `train_taal()` → `forward_backward()`. The notebook reuses the original selector; the scoring cell exposes the same operations outside Dassl. The frozen study is not involved.

In [ ]:
from pathlib import Path
import hashlib, json, random, sys
import torch
from PIL import Image
from IPython.display import display
from collections import Counter

root = Path.home() / 'nola-walkthrough'
upstream = root / 'upstream'
sys.dont_write_bytecode = True
sys.path[:0] = [str(upstream), str(root / '.deps')]
from clip import clip
from utils.data_utils import ds_specific_templates
from utils.model_utils import top_k_indices_per_class2

data = root / 'data/eurosat'
config = dict(images=data/'EuroSAT_RGB', split=data/'split_zhou_EuroSAT.json',
              descriptions=upstream/'descriptions/generic/EuroSAT.json',
              templates=ds_specific_templates['EuroSAT'], seed=42, train_limit=160, test_limit=80, k=4)

import pandas as pd

def anchors_without_labels(scores, class_ids, k):
    if not 1 <= k or scores.ndim != 2 or scores.shape[1] < 2:
        raise ValueError('Expected positive k and a score matrix with at least two classes')
    depth = min(3, scores.shape[1])
    values, ranks = scores.topk(depth, dim=1)
    table = pd.DataFrame({'idx': range(len(scores))})
    for rank in range(depth):
        table[f'c{rank}'] = ranks[:, rank].cpu().tolist()
        table[f'p{rank}'] = values[:, rank].cpu().tolist()
    result, used, missing = [], set(), []
    for cls in class_ids:
        for rank in range(depth):
            candidates = table.loc[table[f'c{rank}']==cls].sort_values(f'p{rank}', ascending=False).head(k)
            if not candidates.empty:
                for i in candidates.idx:
                    if i not in used:
                        result.append((i, cls)); used.add(i)
                break
        else:
            missing.append(cls)
    return result, missing

## 1. Dataset adapter
The shared contract is `train`, `val`, and `test` lists of `[relative image path, class ID, class name]`. IDs index classifier columns. The adapter validates the complete underlying split, then samples each split without using per-image labels. The test sample is held apart from anchor selection. On a new dataset, supply the same schema (or write a small adapter), real class names, a matching descriptions dictionary and useful prompts; the scoring stage does not change.

In [ ]:
def load_dataset(cfg):
    manifest = cfg['split'].read_bytes()
    splits = json.loads(manifest)
    if set(splits) != {'train','val','test'}:
        raise ValueError('Expected train/val/test splits')
    classes = {}
    seen = set()
    image_root = cfg['images'].resolve()
    for split_name, rows in splits.items():
        for relative, label, name in rows:
            path = (image_root / relative).resolve()
            if not path.is_relative_to(image_root) or path in seen or not path.is_file():
                raise ValueError(f'Invalid, repeated or missing image: {relative}')
            seen.add(path)
            if label is None and split_name=='train':
                continue
            if not isinstance(label,int) or not isinstance(name,str):
                raise ValueError('Provide integer labels and class names outside unlabeled train rows')
            if classes.setdefault(label, name) != name:
                raise ValueError(f'Conflicting name for class ID {label}')
    if sorted(classes) != list(range(len(classes))) or len(set(classes.values())) != len(classes):
        raise ValueError('Class IDs must be dense from zero and names unique')
    ordered = [classes[i] for i in range(len(classes))]
    descriptions = json.loads(cfg['descriptions'].read_text())
    if set(ordered) != set(descriptions) or any(not texts for texts in descriptions.values()):
        raise ValueError('Supply descriptions for exactly the manifest class names')
    print('Split SHA256:', hashlib.sha256(manifest).hexdigest())
    return splits, ordered, descriptions

splits, names, descriptions = load_dataset(config)
print('Verified files:', {key:len(rows) for key,rows in splits.items()})
print('Classifier columns:', names)
def sample_split(rows, limit, seed):
    return [rows[i] for i in sorted(random.Random(seed).sample(range(len(rows)), min(limit,len(rows))))]
train = sample_split(splits['train'], config['train_limit'], config['seed'])
test = sample_split(splits['test'], config['test_limit'], config['seed'])
print('Walkthrough sample:', len(train), 'adaptation images and', len(test), 'held-out images')
with Image.open(config['images']/train[0][0]) as example:
    display(example.resize((224,224)))
print('Example file:', train[0][0])

## 2. CDE classifier on real satellite images
`gen_emb()` normalizes individual text vectors, averages them per class, and normalizes the class mean. `forward_with_text_desc()` computes scaled cosine logits. With ten classes, one image creates ten logits. Softmax creates scores relative to the ten candidates; it is not calibrated confidence. We also score a six-template-only baseline on the same held-out sample. Neither result is a paper reproduction.

In [ ]:
model, width, preprocess = clip.load('ViT-B/32', device='cuda', download_root=str(root/'weights'))
model.eval()
@torch.inference_mode()
def text_matrix(include_descriptions=True):
    cols = []
    for name in names:
        texts = (descriptions[name] if include_descriptions else []) + [t.format(name) for t in config['templates']]
        e = model.encode_text(clip.tokenize(texts).cuda())
        m = (e/e.norm(dim=-1, keepdim=True)).mean(dim=0)
        cols.append(m/m.norm())
    return torch.stack(cols, dim=1)

@torch.inference_mode()
def image_features(rows, batch_size=16):
    out = []
    for start in range(0,len(rows),batch_size):
        ims = []
        for rel,_,_ in rows[start:start+batch_size]:
            with Image.open(config['images']/rel) as im:
                ims.append(preprocess(im))
        f = model.encode_image(torch.stack(ims).cuda())
        out.append(f/f.norm(dim=-1,keepdim=True))
    return torch.cat(out)

cde = text_matrix()
scale = model.logit_scale.detach().exp()
train_f, test_f = image_features(train), image_features(test)
probs = (scale * train_f @ cde).softmax(dim=1).cpu()
assert probs.shape == (len(train),len(names)) and torch.isfinite(probs).all()
y_train = None if any(row[1] is None for row in train) else torch.tensor([row[1] for row in train]); y_test = torch.tensor([row[1] for row in test])
print('Text weights:', tuple(cde.shape), '| adaptation features:', tuple(train_f.shape), '| scores:', tuple(probs.shape))
for mode, w in [('Six templates only',text_matrix(False)), ('Descriptions + templates',cde)]:
    pred = (scale * test_f @ w).argmax(dim=1).cpu()
    print(f'{mode}: {(pred==y_test).sum().item()}/{len(test)} held-out correct')
for value,idx in zip(*probs[0].topk(3)):
    print(names[idx.item()], round(value.item(),3))

## 3. Original anchor selector

**Saved pilot interpretation:** On the same 80 held-out images, adding descriptions increased the correct count from 32 to 35. This small comparison demonstrates how to hold the image set fixed; it does not establish a reliable improvement.
`select_top_k()` passes the score matrix, IDs, and dataset labels to `top_k_indices_per_class2()`. The original function reads labels to enumerate classes present and print diagnostics; individual assignments come from scores. If there are no top-1 candidates for a class, it checks second choices, then third choices; it returns fewer than `k` when necessary and deduplicates image IDs. Returned `prob1` and its own aggregate accuracy print can refer to the pre-fallback prediction. Below we separately compute correctness against **returned** pseudo-labels and show both scores. Unlabeled custom datasets cannot call this helper unmodified: pass an explicit class vocabulary through a separately tested equivalent selector and omit accuracy diagnostics rather than providing fake labels.

In [ ]:
top1 = probs.argmax(dim=1)
if y_train is not None and len(names)>=3:
    anchor_ids, returned_prob1, pseudo = top_k_indices_per_class2(
        dict(total_emb=probs, idxs=torch.arange(len(train)), labels=y_train), config['k'])
else:
    available, missing_classes = anchors_without_labels(probs, range(len(names)), config['k'])
    if not available:
        raise ValueError('No anchors selected')
    anchor_ids, pseudo = (torch.tensor(ids,dtype=torch.long) for ids in zip(*available))
    returned_prob1 = probs.max(dim=1).values[anchor_ids]
    print('Classes with no candidates:',[names[i] for i in missing_classes])
assert len(set(anchor_ids.tolist())) == len(anchor_ids)
if y_train is not None:
    print('Returned anchor correctness:', (y_train[anchor_ids]==pseudo).sum().item(), '/', len(anchor_ids))
print('Reassigned by fallback:', (top1[anchor_ids]!=pseudo).sum().item())
for i,name in enumerate(names):
    print(f'{name:28} top-1 candidates={int((top1==i).sum()):3d} anchors={int((pseudo==i).sum()):2d}')
for pos in range(min(4,len(anchor_ids))):
    i,j = anchor_ids[pos].item(),pseudo[pos].item()
    print(f'{train[i][0]} | pseudo={names[j]} | audit={train[i][2]} | assigned score={probs[i,j]:.3f} | returned prob1={returned_prob1[pos]:.3f}')
print('Handoff: image paths indexed by anchor_ids + pseudo-label tensor', tuple(pseudo.shape))

### 3.1 Custom data without image labels

**Saved pilot interpretation:** CLIP got 90/160 adaptation predictions right. The selector returned 25 correct labels among 34 anchors. This higher precision comes at the cost of coverage: Pasture received one anchor, and Herbaceous Vegetation needed the third-choice fallback. Look at returned labels, not the helper’s stale pre-fallback accuracy.
The released selector gets its class set from the labels tensor. This adapter takes that set from the declared vocabulary instead, but uses the same rank fallback, sort, and first-occurrence deduplication. If a declared class has no candidate even at rank 3, it reports and skips the class rather than raising like the original. This is a porting choice, not a change to the EuroSAT baseline. The check below compares both paths on the same observed scores. Unknown adaptation labels may be represented by `null` IDs; use explicit class IDs and names from a labeled evaluation split, and omit the accuracy audits when labels are absent.

In [ ]:
if y_train is not None and len(names)>=3 and set(y_train.tolist())==set(range(len(names))):
    label_free, absent = anchors_without_labels(probs, range(len(names)), config['k'])
    assert label_free == list(zip(anchor_ids.tolist(), pseudo.tolist())) and not absent
    print(f'Explicit-vocabulary adapter matches original on all {len(label_free)} selected image/label pairs.')

## 4. DINO teacher: frozen image encoder + a trained head
The released `Taal` uses DINO ViT-B/16 and `AdapterMLP` (768 → 256 → class count). This notebook reuses that exact head definition, the original test transform, Adam and cross-entropy. The walkthrough uses three epochs and a small subset; it is not the paper's full schedule. The teacher sees **CLIP's labels for the selected images**, not their folder labels. After fitting, it supplies hard labels for the whole adaptation sample. DINO and CLIP are staged to avoid holding both in GPU memory during student training.

In [ ]:
import ast, math
from torch import nn
from utils.model_utils import te_transform, tr_transforms

trainer_path = upstream/'trainers/NoLA.py'
trainer_tree = ast.parse(trainer_path.read_text())
# Use the upstream class definition without starting Dassl or W&B.
adapter = next(x for x in trainer_tree.body if isinstance(x, ast.ClassDef) and x.name=='AdapterMLP')
ns = {'nn': nn}
exec(compile(ast.Module(body=[adapter],type_ignores=[]),str(trainer_path),'exec'),ns)
weights = root/'weights/dino_vitbase16_pretrain.pth'
assert weights.stat().st_size == 343242485, 'Download the complete official DINO weights first'
# DINO and NoLA both call their utility module "utils"; isolate the local hub import.
old_utils = {k:v for k,v in list(sys.modules.items()) if k=='utils' or k.startswith('utils.')}
for k in old_utils:
    sys.modules.pop(k)
try:
    dino = torch.hub.load(str(root/'dino-source'),'dino_vitb16',source='local',pretrained=False)
finally:
    for k in list(sys.modules):
        if k=='utils' or k.startswith('utils.'):
            sys.modules.pop(k)
    sys.modules.update(old_utils)
dino.load_state_dict(torch.load(weights,map_location='cpu',weights_only=True),strict=True)
dino.eval().requires_grad_(False).cuda()

@torch.no_grad()
def dino_features(rows, batch_size=8):
    out=[]
    for start in range(0,len(rows),batch_size):
        batch=[]
        for rel,_,_ in rows[start:start+batch_size]:
            with Image.open(config['images']/rel) as im:
                batch.append(te_transform(im))
        out.append(dino(torch.stack(batch).cuda()).cpu())
    return torch.cat(out)

dino_train, dino_test = dino_features(train), dino_features(test)
dino.cpu(); del dino; torch.cuda.empty_cache()
assert dino_train.shape == (len(train),768)
torch.manual_seed(config['seed'])
head = ns['AdapterMLP'](num_classes=len(names),input_size=768,hidden_size=256)
opt = torch.optim.Adam(head.parameters())
for epoch in range(3):
    for batch_ids in torch.randperm(len(anchor_ids)).split(16):
        opt.zero_grad()
        loss = nn.functional.cross_entropy(head(dino_train[anchor_ids[batch_ids]]), pseudo[batch_ids])
        loss.backward(); opt.step()
    print(f'Teacher epoch {epoch+1}: last batch loss {loss.item():.3f}')
with torch.no_grad():
    teacher_labels = head(dino_train).argmax(dim=1)
    teacher_test = head(dino_test).argmax(dim=1)
print('DINO features:', tuple(dino_train.shape), '| teacher labels:', tuple(teacher_labels.shape))
print('Teacher held-out correct:', (teacher_test==y_test).sum().item(), '/', len(test))
print('Example teacher label for', train[0][0], ':', names[teacher_labels[0].item()])

## 5. Prompted CLIP student: one teaching pass

**Saved pilot interpretation:** The head turned each 768-feature DINO vector into a class label. It got 40/80 held-out images right after three epochs. That is not the same as fitting its pseudo-labels; the held-out labels were not used to train it.
The original `forward_backward()` gets the hard teacher label from a weak view, then trains on a strongly augmented view of the same image. Here we cache those labels because the released weak transform is deterministic and the teacher is frozen. We use the upstream strong transform, classifier initialization from CDE, and original prompt insertion / forward methods. This is one short pass with an illustrative optimizer learning rate, not the full Dassl training loop or checkpoint-selection protocol. A loss drop by itself does not prove that the student is more accurate.

In [ ]:
from functools import reduce
from operator import mul
loss_path = upstream/'utils/utils.py'
loss_tree = ast.parse(loss_path.read_text())
loss_definitions = [x for x in loss_tree.body if getattr(x,'name','') in {'linear_combination','reduce_loss','LabelSmoothingCrossEntropy'}]
loss_ns = {'nn':nn,'F':nn.functional, 'torch':torch}
exec(compile(ast.Module(body=loss_definitions,type_ignores=[]),str(loss_path),'exec'),loss_ns)
LabelSmoothingCrossEntropy = loss_ns['LabelSmoothingCrossEntropy']

net = next(x for x in trainer_tree.body if isinstance(x, ast.ClassDef) and x.name=='NoLAUFT')
wanted = {'incorporate_prompt','patch_embeddings','embeddings_after_prompts','forward_aug_with_prompts'}
ns2 = {'torch': torch, 'nn': nn}
exec(compile(ast.Module(body=[x for x in net.body if isinstance(x,ast.FunctionDef) and x.name in wanted],type_ignores=[]),str(trainer_path),'exec'),ns2)
class Student(nn.Module):
    incorporate_prompt = ns2['incorporate_prompt']
    patch_embeddings = ns2['patch_embeddings']
    embeddings_after_prompts = ns2['embeddings_after_prompts']
    forward = ns2['forward_aug_with_prompts']
    def __init__(self, clip_model, weights):
        super().__init__()
        self.clip_model = clip_model.float()
        self.prompt_proj = nn.Identity(); self.prompt_dropout = nn.Dropout(0.0)
        self.prompt_embeddings = nn.Parameter(torch.empty(1,50,768))
        val = math.sqrt(6. / (3 * reduce(mul,(16,16)) + 768))
        nn.init.uniform_(self.prompt_embeddings,-val,val)
        self.cde_adapter = nn.Sequential(nn.Linear(weights.shape[0],weights.shape[1],bias=False))
        with torch.no_grad():
            self.cde_adapter[0].weight.copy_(weights.T.float())
        for name, par in self.named_parameters():
            par.requires_grad_('prompt_embeddings' in name or 'cde_adapter' in name or
                               ('visual' in name and ('ln' in name or 'bn' in name)))

torch.manual_seed(config['seed']); random.seed(config['seed'])
student = Student(model,cde).cuda().eval()
selected = [(name,p) for name,p in student.named_parameters() if p.requires_grad]
optimizer = torch.optim.AdamW([
    {'params':[p for name,p in selected if 'bias' not in name], 'weight_decay':0.01},
    {'params':[p for name,p in selected if 'bias' in name], 'weight_decay':0.0}],lr=0.001)
criterion = LabelSmoothingCrossEntropy()

def image_batch(rows, strong=False):
    xs=[]
    for path,_,_ in rows:
        with Image.open(config['images']/path) as im:
            rgb=im.convert('RGB')
            xs.append(tr_transforms(rgb)[1] if strong else te_transform(rgb))
    return torch.stack(xs).cuda()

@torch.no_grad()
def student_eval():
    preds = [student(image_batch(test[i:i+8])).argmax(dim=1).cpu() for i in range(0,len(test),8)]
    return int((torch.cat(preds)==y_test).sum())

initial = student.prompt_embeddings.detach().clone(); before = student_eval()
losses=[]
for ids in torch.randperm(len(train)).split(8):
    x = image_batch([train[i] for i in ids.tolist()],strong=True)
    optimizer.zero_grad()
    loss = criterion(student(x),teacher_labels[ids].cuda())
    loss.backward(); optimizer.step(); losses.append(loss.item())
change = (initial-student.prompt_embeddings.detach()).norm().item()
assert change>0 and all(math.isfinite(x) for x in losses)
print('Visual prompt shape:', tuple(student.prompt_embeddings.shape), '| updated parameter count:',sum(p.numel() for _,p in selected))
print(f'Student ran {len(losses)} updates; first/last loss {losses[0]:.3f}/{losses[-1]:.3f}; prompt change L2 {change:.4f}')
print('Held-out correct before/after student update:', before, '/', len(test), '→', student_eval(), '/', len(test))

## Porting order

**Saved pilot interpretation:** The prompt changed and the prompted model improved from 11 to 19 correct out of 80, but still fell short of the unprompted CDE classifier’s 35. This verifies that the student update ran. It does not demonstrate a NoLA accuracy gain.
First check the full EuroSAT run separately from this pilot. Next reuse the adapter on another reported benchmark such as Oxford Pets, then ImageNet when data access and compute are in place. For a newer dataset, record the release date, license, vocabulary and split; a newer release does not prove CLIP or DINO never saw related images. A different task such as detection, multi-label recognition or segmentation changes the method, not just the adapter.